In [0]:
# =============================================================================
# Notebook 05 — Target Definition, Log-Transform, and GroupKFold Validation
# Objective: Define the prediction targets (log1p H2 rate, band gap),
#            remove zero-variance features, validate the GroupKFold split
#            strategy, compute ICC(DOI) to set the theoretical R² ceiling,
#            and write the final analysis-ready tables.
#
# Inputs : ml_perovskites.master, ml_perovskites.X_exp, ml_perovskites.X_material
# Outputs: ml_perovskites.y_targets   — target columns + grouping key
#          ml_perovskites.X_exp_clean  — L3 features after zero-var filter
#          ml_perovskites.X_mat_clean  — L1 features after zero-var filter
#          /tmp/05_target_distribution.png
#          /tmp/05_group_distribution.png
# Seed   : 42
# Author : Bernardo Araldi da Silva | UFSC | 2026
# =============================================================================

# =============================================================================
# SECTION 1 — Imports and configuration
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import GroupKFold
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_SPLITS = 5   # GroupKFold folds

print("Section 1 complete: imports and configuration loaded.")

# =============================================================================
# SECTION 2 — Load tables
# =============================================================================

print("\n--- Loading tables ---")

master_pd = spark.table("ml_perovskites.master").toPandas()
xexp_pd   = spark.table("ml_perovskites.X_exp").toPandas()
xmat_pd   = spark.table("ml_perovskites.X_material").toPandas()

print(f"master shape     : {master_pd.shape}")
print(f"X_exp shape      : {xexp_pd.shape}")
print(f"X_material shape : {xmat_pd.shape}")

# =============================================================================
# SECTION 3 — Target columns
# =============================================================================

print("\n--- Defining target columns ---")

# Primary target: H2 production rate in µmol·g⁻¹·h⁻¹
# Secondary target: band gap in eV (for multi-task learning later)
# Grouping key: doi (for GroupKFold — no same-paper rows in train+test)

target_cols = ["source_row_id", "doi",
               "h2_rate_umol_g_h", "bandgap_eV",
               "family_proposed", "domain"]

# Check availability
for col in target_cols:
    present = col in master_pd.columns
    if not present:
        print(f"  MISSING column: {col}")

y_raw = master_pd[[c for c in target_cols if c in master_pd.columns]].copy()

# Coerce numeric targets
y_raw["h2_rate_umol_g_h"] = pd.to_numeric(y_raw["h2_rate_umol_g_h"], errors="coerce")
y_raw["bandgap_eV"]        = pd.to_numeric(y_raw["bandgap_eV"],        errors="coerce")

print(f"\nh2_rate_umol_g_h — total: {len(y_raw)}, "
      f"non-null: {y_raw['h2_rate_umol_g_h'].notna().sum()}, "
      f"null: {y_raw['h2_rate_umol_g_h'].isna().sum()}")
print(f"bandgap_eV       — total: {len(y_raw)}, "
      f"non-null: {y_raw['bandgap_eV'].notna().sum()}, "
      f"null: {y_raw['bandgap_eV'].isna().sum()}")

# =============================================================================
# SECTION 4 — Join targets with X_exp (inner on source_row_id)
# =============================================================================

print("\n--- Joining targets with feature matrix ---")

# X_exp already contains 682 parse_ok rows
# Master has 691 rows; 9 have parse_ok=0 and are not in X_exp
# Inner join keeps only rows present in both → 682 rows maximum

df = xexp_pd.merge(
    y_raw[["source_row_id", "h2_rate_umol_g_h", "bandgap_eV"]],
    on="source_row_id",
    how="inner"
)

print(f"Joined shape: {df.shape}")
print(f"  h2_rate non-null after join: {df['h2_rate_umol_g_h'].notna().sum()}")
print(f"  bandgap  non-null after join: {df['bandgap_eV'].notna().sum()}")

# =============================================================================
# SECTION 5 — Log-transform the H2 rate target
# =============================================================================

print("\n--- Applying log1p transform to H2 rate ---")

# Rationale: H2 rate spans ~6 orders of magnitude (0 to 10^5 µmol/g/h).
# log1p(y) compresses the range, reduces heteroscedasticity, and makes
# residuals more normally distributed — standard practice for skewed rate data.
# log1p handles zero values (log1p(0) = 0).

df["log1p_h2_rate"] = np.log1p(df["h2_rate_umol_g_h"].fillna(0))

print(f"Original H2 rate stats (µmol/g/h):")
print(df["h2_rate_umol_g_h"].describe().to_string())
print(f"\nlog1p H2 rate stats:")
print(df["log1p_h2_rate"].describe().to_string())

# =============================================================================
# SECTION 6 — Target distribution plots
# =============================================================================

print("\n--- Plotting target distributions ---")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original scale
h2_vals = df["h2_rate_umol_g_h"].dropna()
axes[0].hist(h2_vals, bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("H₂ rate (µmol·g⁻¹·h⁻¹)")
axes[0].set_ylabel("Count")
axes[0].set_title("H₂ rate — original scale")

# Log scale
axes[1].hist(np.log10(h2_vals[h2_vals > 0] + 1), bins=50,
             color="darkorange", edgecolor="white", linewidth=0.3)
axes[1].set_xlabel("log₁₀(H₂ rate + 1)")
axes[1].set_ylabel("Count")
axes[1].set_title("H₂ rate — log₁₀ scale")

# Band gap
bg_vals = df["bandgap_eV"].dropna()
axes[2].hist(bg_vals, bins=40, color="seagreen", edgecolor="white", linewidth=0.3)
axes[2].set_xlabel("Band gap (eV)")
axes[2].set_ylabel("Count")
axes[2].set_title("Band gap distribution")

plt.tight_layout()
plt.savefig("/tmp/05_target_distribution.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/05_target_distribution.png")

# =============================================================================
# SECTION 7 — Zero-variance feature removal
# =============================================================================

print("\n--- Removing zero-variance features ---")

# Identify feature columns in X_exp (exclude id/metadata/target cols)
META_COLS = {"source_row_id", "base_compound", "family_proposed", "formula_reduced",
             "n_elements", "n_A_elements", "n_B_elements",
             "A_site_elements", "B_site_elements", "parse_ok",
             "doi", "Compound_original", "domain",
             "h2_rate_umol_g_h", "bandgap_eV", "log1p_h2_rate"}

feature_cols_exp = [c for c in df.columns if c not in META_COLS]
feature_cols_mat = [c for c in xmat_pd.columns
                    if c not in META_COLS and c in df.columns]

# Zero-variance check: std == 0 OR all-NaN
def find_zero_var_cols(df_sub, cols):
    drop = []
    for col in cols:
        if col not in df_sub.columns:
            continue
        std = df_sub[col].std()
        if std == 0 or pd.isna(std):
            drop.append(col)
    return drop

zv_exp = find_zero_var_cols(df, feature_cols_exp)
zv_mat = find_zero_var_cols(df, feature_cols_mat)

print(f"Zero-variance cols in L3 (X_exp)     : {len(zv_exp)}")
if zv_exp:
    print(f"  Dropping: {zv_exp}")

print(f"Zero-variance cols in L1 (X_material) : {len(zv_mat)}")
if zv_mat:
    print(f"  Dropping: {zv_mat}")

feature_cols_exp_clean = [c for c in feature_cols_exp if c not in zv_exp]
feature_cols_mat_clean = [c for c in feature_cols_mat if c not in zv_mat]

print(f"\nFeatures remaining — L3: {len(feature_cols_exp_clean)} | L1: {len(feature_cols_mat_clean)}")

# Build clean feature frames (keep source_row_id and doi as join/grouping keys)
x_exp_clean = df[["source_row_id", "doi"] + feature_cols_exp_clean].copy()
x_mat_clean = df[["source_row_id", "doi"] + feature_cols_mat_clean].copy()

# =============================================================================
# SECTION 8 — GroupKFold split validation
# =============================================================================

print("\n--- Validating GroupKFold split strategy ---")

# Groups are defined by doi — all rows from the same paper stay together.
# This prevents data leakage between papers (same experimental setup).

groups = df["doi"].values
gkf    = GroupKFold(n_splits=N_SPLITS)

fold_stats = []
for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(df, groups=groups)):
    train_dois = set(df.iloc[train_idx]["doi"].unique())
    val_dois   = set(df.iloc[val_idx]["doi"].unique())
    overlap    = train_dois & val_dois

    train_h2  = df.iloc[train_idx]["h2_rate_umol_g_h"].dropna()
    val_h2    = df.iloc[val_idx]["h2_rate_umol_g_h"].dropna()

    fold_stats.append({
        "fold":          fold_idx + 1,
        "n_train":       len(train_idx),
        "n_val":         len(val_idx),
        "n_train_dois":  len(train_dois),
        "n_val_dois":    len(val_dois),
        "doi_overlap":   len(overlap),
        "train_h2_med":  train_h2.median(),
        "val_h2_med":    val_h2.median(),
    })

fold_df = pd.DataFrame(fold_stats)
print(fold_df.to_string(index=False))

max_overlap = fold_df["doi_overlap"].max()
print(f"\nMax DOI overlap across folds: {max_overlap}")
print("  PASS — no leakage" if max_overlap == 0 else "  FAIL — DOI overlap detected!")

# Distribution of validation H2 rate across folds
fig2, ax2 = plt.subplots(figsize=(8, 4))
for fold_idx, (_, val_idx) in enumerate(gkf.split(df, groups=groups)):
    vals = np.log1p(df.iloc[val_idx]["h2_rate_umol_g_h"].fillna(0))
    ax2.hist(vals, bins=30, alpha=0.5, label=f"Fold {fold_idx+1}")
ax2.set_xlabel("log1p(H₂ rate)")
ax2.set_ylabel("Count")
ax2.set_title(f"GroupKFold validation set distributions ({N_SPLITS} folds)")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/05_group_distribution.png", dpi=120, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/05_group_distribution.png")

# =============================================================================
# SECTION 9 — ICC(DOI): theoretical R² ceiling for material-only models
# =============================================================================

print("\n--- Computing ICC(DOI) — theoretical R² ceiling ---")

# ICC (Intraclass Correlation Coefficient) by DOI decomposes variance into:
#   Between-DOI component: variation explained by compound identity
#   Within-DOI component : variation from different experimental conditions
#
# ICC(DOI) ≈ R² ceiling for a model that sees only material descriptors.
# A high ICC means the compound largely determines the rate (good for ML).
# A low ICC means experimental conditions dominate (material-only model is limited).

h2_log = df[["doi", "log1p_h2_rate"]].dropna()
grand_mean = h2_log["log1p_h2_rate"].mean()
grand_var  = h2_log["log1p_h2_rate"].var()

# Between-group sum of squares
group_means = h2_log.groupby("doi")["log1p_h2_rate"].mean()
group_sizes = h2_log.groupby("doi")["log1p_h2_rate"].count()
ss_between  = ((group_means - grand_mean) ** 2 * group_sizes).sum()
df_between  = len(group_means) - 1

# Within-group sum of squares
ss_within   = h2_log.groupby("doi")["log1p_h2_rate"].apply(
    lambda x: ((x - x.mean()) ** 2).sum()
).sum()
df_within   = len(h2_log) - len(group_means)

# Mean squares
ms_between = ss_between / df_between if df_between > 0 else 0
ms_within  = ss_within  / df_within  if df_within  > 0 else 0

# Average group size (harmonic mean for unbalanced design)
n_total = len(h2_log)
n_groups = len(group_means)
n0 = (n_total - (group_sizes ** 2).sum() / n_total) / (n_groups - 1) if n_groups > 1 else 1

# ICC(1) — one-way random effects
sigma2_between = max((ms_between - ms_within) / n0, 0.0)
sigma2_within  = ms_within

icc = sigma2_between / (sigma2_between + sigma2_within) if (sigma2_between + sigma2_within) > 0 else 0

print(f"  Grand mean (log1p H2 rate)    : {grand_mean:.4f}")
print(f"  Grand variance                 : {grand_var:.4f}")
print(f"  MS_between                     : {ms_between:.4f}")
print(f"  MS_within                      : {ms_within:.4f}")
print(f"  sigma2_between (compound)      : {sigma2_between:.4f}")
print(f"  sigma2_within  (experimental)  : {sigma2_within:.4f}")
print(f"  n0 (harmonic mean group size)  : {n0:.2f}")
print(f"\n  ICC(DOI) = {icc:.4f}")
print(f"  Interpretation: {icc*100:.1f}% of H2 rate variance is attributable to")
print(f"  compound identity. Theoretical R² ceiling for material-only models: {icc:.2f}")

if icc >= 0.50:
    print("  → Material descriptors alone should be informative (ICC ≥ 0.50).")
elif icc >= 0.30:
    print("  → Moderate compound effect; experimental features essential for good R².")
else:
    print("  → Weak compound effect; experimental conditions dominate variance.")

# =============================================================================
# SECTION 10 — Build and write y_targets table
# =============================================================================

print("\n--- Writing y_targets table ---")

y_targets = df[["source_row_id", "doi", "family_proposed",
                "h2_rate_umol_g_h", "log1p_h2_rate", "bandgap_eV"]].copy()

# Add activity tier label (for stratified diagnostics — NOT for training)
def activity_tier(rate):
    if pd.isna(rate):
        return "unknown"
    if rate == 0:
        return "zero"
    if rate < 10:
        return "very_low"
    if rate < 100:
        return "low"
    if rate < 1000:
        return "medium"
    if rate < 10000:
        return "high"
    return "extreme"

y_targets["activity_tier"] = y_targets["h2_rate_umol_g_h"].apply(activity_tier)

print("Activity tier distribution:")
print(y_targets["activity_tier"].value_counts().to_string())

y_spark = spark.createDataFrame(
    y_targets.where(pd.notnull(y_targets), other=None)
)
(y_spark
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.y_targets"))

n_y = spark.table("ml_perovskites.y_targets").count()
print(f"\nRows written to ml_perovskites.y_targets: {n_y}")

# =============================================================================
# SECTION 11 — Write clean feature tables
# =============================================================================

print("\n--- Writing clean feature tables ---")

def write_delta(df_pd, table_name):
    df_clean = df_pd.where(pd.notnull(df_pd), other=None)
    spark_df = spark.createDataFrame(df_clean)
    (spark_df
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name))
    n = spark.table(table_name).count()
    print(f"  {table_name}: {n} rows, {df_pd.shape[1]} columns")

write_delta(x_exp_clean, "ml_perovskites.X_exp_clean")
write_delta(x_mat_clean, "ml_perovskites.X_mat_clean")

# =============================================================================
# SECTION 12 — Acceptance criteria checks
# =============================================================================

print("\n" + "="*60)
print("ACCEPTANCE CRITERIA CHECKS")
print("="*60)

# CHECK 1: y_targets row count
n_y_check = spark.table("ml_perovskites.y_targets").count()
print(f"\n[CHECK 1] y_targets row count: {n_y_check} (expected 682)")
print("  PASS" if n_y_check == 682 else f"  FAIL — got {n_y_check}")

# CHECK 2: No DOI overlap in any fold
print(f"\n[CHECK 2] GroupKFold DOI overlap max: {max_overlap}")
print("  PASS" if max_overlap == 0 else "  FAIL — leakage detected")

# CHECK 3: ICC is positive and finite
print(f"\n[CHECK 3] ICC(DOI): {icc:.4f}")
print("  PASS" if 0 < icc <= 1 else "  WARNING — ICC out of expected range")

# CHECK 4: Zero-variance columns removed
all_zv = set(zv_exp) | set(zv_mat)
print(f"\n[CHECK 4] Zero-variance columns removed: {len(all_zv)}")
print(f"  {sorted(all_zv)}" if all_zv else "  (none)")
print("  PASS")

# CHECK 5: log1p_h2_rate has no NaN
n_nan_log = y_targets["log1p_h2_rate"].isna().sum()
print(f"\n[CHECK 5] NaN in log1p_h2_rate: {n_nan_log}")
print("  PASS" if n_nan_log == 0 else f"  WARNING — {n_nan_log} NaN values")

print("\n" + "="*60)

# =============================================================================
# SECTION 13 — Summary
# =============================================================================

print("\n" + "="*60)
print("NOTEBOOK 05 COMPLETE")
print("="*60)
print(f"Output tables:")
print(f"  ml_perovskites.y_targets   : {n_y} rows")
print(f"  ml_perovskites.X_exp_clean : {len(x_exp_clean)} rows, "
      f"{len(feature_cols_exp_clean)} features")
print(f"  ml_perovskites.X_mat_clean : {len(x_mat_clean)} rows, "
      f"{len(feature_cols_mat_clean)} features")
print(f"\nKey diagnostics:")
print(f"  ICC(DOI)                   : {icc:.4f}")
print(f"  Theoretical R² ceiling     : {icc:.2f} (material-only models)")
print(f"  Zero-variance cols removed : {len(all_zv)}")
print(f"  GroupKFold folds           : {N_SPLITS} (no DOI overlap confirmed)")
print(f"\nTargets:")
print(f"  Primary   : log1p_h2_rate (log1p of µmol·g⁻¹·h⁻¹)")
print(f"  Secondary : bandgap_eV")
print(f"  Grouping  : doi (GroupKFold key — NOT a feature)")
print(f"\nNext step: Notebook 06 — Baseline models (Ridge, RF, XGBoost)")
print(f"           under GroupKFold CV on L1, L2, and L3 feature sets.")